In [0]:
from pyspark.sql import functions as F

# 1. load from Bronze
sensors_df = spark.table("crude_ops.bronze.drilling_raw")
sap_df = spark.table("crude_ops.bronze.sap_operations")

# 2. dynamic load data anchor 
base_date_val = sap_df.select(F.to_date(F.min("start_time"))).collect()[0][0]
print(f"Base Date identified: {base_date_val}")

# 3. Transformation  Double (Excel format) інто Timestamp
# formula: (Excel_Value - 25569) * 86400 (seconds per day)
silver_sensors_df = sensors_df.withColumn(
    "event_timestamp",
    ((F.col("TIME_1900") - 25569) * 86400).cast("timestamp")
)

# 4. INTERVAL JOIN: connect sensors to SAP operations
# for each sensor recors find interval in SAP
silver_enriched_df = silver_sensors_df.join(
    sap_df,
    (silver_sensors_df.event_timestamp >= sap_df.start_time) & 
    (silver_sensors_df.event_timestamp <= sap_df.end_time),
    "left"
)

# 5. clean and selection (use names, previously found via Spark)
silver_enriched_df = silver_sensors_df.join(
    sap_df,
    (silver_sensors_df.event_timestamp >= sap_df.start_time) & 
    (silver_sensors_df.event_timestamp <= sap_df.end_time),
    "left" 
)

final_silver_df = silver_enriched_df.select(
    F.col("event_timestamp"),
    F.col("well_id"),
    F.col("op").alias("operation"),
    "phase",
    "DEPTH",
    "GR"
)

# 6. Load to Silver
final_silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("crude_ops.silver.drilling_enriched")

print(f"Sensors Time Range: {silver_sensors_df.select(F.min('event_timestamp'), F.max('event_timestamp')).collect()}")
print(f"SAP Time Range:     {sap_df.select(F.min('start_time'), F.max('end_time')).collect()}")

# check verification
print(f"Loading complete. Total rows: {final_silver_df.count()}")
display(spark.table("crude_ops.silver.drilling_enriched").orderBy("event_timestamp").limit(10))